## Transform Customer Data
1. Remove records with null customer id
2. Remove exact duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the columns to the correct data type
5. Write transfomed data to silver schema

### 1. Remove records with null customer_id

In [0]:
SELECT * from gizmobox.bronze.v_customers
where customer_id IS NOT NULL
ORDER BY customer_id;

### 2. Remove exact duplicate records

In [0]:
SELECT DISTINCT *
FROM
gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;


In [0]:
-- one wrong way of removing duplicates 

SELECT customer_id,
        MAX(created_timestamp),
        MAX(customer_name),
        MAX(date_of_birth),
        MAX(email),
        MAX(member_since),
        MAX(telephone)
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
GROUP BY customer_id
ORDER BY customer_id;

### 3. Remove duplicate records based on created_timestamp

In [0]:
-- correct way to do this
--lets create a temp view

CREATE OR REPLACE TEMPORARY view v_customer_distinct
AS
select distinct *
from gizmobox.bronze.v_customers
where customer_id is not NULL
order by customer_id


In [0]:
-- get the highest created time stamp for customer id

select customer_id,
        max(created_timestamp) as max_created_timestamp
    FROM v_customer_distinct
    group by customer_id



In [0]:
-- lets create a common table expression // it aslo called as inline view sometimes
with cte_max as 
(
    SELECT customer_id,
        max(created_timestamp) as max_created_timestamp
        from v_customer_distinct
        group by customer_id
)
select t.*
from v_customer_distinct t
join cte_max m 
on t.customer_id = m.customer_id
and t.created_timestamp = m.max_created_timestamp;


### 4. CAST the columns values to the correct data type

In [0]:
with cte_max as
(
    select customer_id,
    max(created_timestamp) as max_created_timestamp
    from v_customer_distinct
    group by customer_id
)
select CAST(t.created_timestamp as timestamp) as created_timestamp,
        t.customer_id,
        t.customer_name,
        CAST(t.date_of_birth as date) as date_of_birth,
        t.email,
        CAST(t.member_since as date) as member_since,
        t.telephone
from v_customer_distinct t
join cte_max m 
on t.customer_id = m.customer_id
and t.created_timestamp = m.max_created_timestamp;

### 5. Write Data to a Delta Table

In [0]:
-- c tast tsble

create table gizmobox.silver.customers
as
with cte_max as
(
    select customer_id, 
    max(created_timestamp) as max_created_timestamp
    from v_customer_distinct
    group by customer_id
)
select cast(t.created_timestamp as timestamp) as created_timestamp,
        t.customer_id,
        t.customer_name,
        cast(t.date_of_birth as date) as date_of_birth,
        t.email,
        cast(t.member_since as date) as member_since,
        t.telephone
from v_customer_distinct t
join cte_max m 
on t.customer_id = m.customer_id
and t.created_timestamp = m.max_created_timestamp;

In [0]:
select * from gizmobox.silver.customers

In [0]:
DESCRIBE EXTENDED gizmobox.silver.customers;